# Data Cleaning and Validation

This notebook validates the three existing cleaned CSVs independently. It reads from paths relative to the project root, performs any column-name normalization only in memory, and does not overwrite source files, invent values or dates, merge datasets, or build models.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

working_directory = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "data" / "cleaned").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find the project root containing data/cleaned from "
        f"{working_directory}"
    )

cleaned_dir = project_root / "data" / "cleaned"
dataset_paths = {
    "Infrastructure": cleaned_dir / "infrastructure_state_year.csv",
    "Budget": cleaned_dir / "nhm_budget_long_corrected.csv",
    "HMIS": cleaned_dir / "hmis_long_normalized.csv",
}
missing_paths = [path for path in dataset_paths.values() if not path.is_file()]
if missing_paths:
    missing_list = "\n".join(f"- {path}" for path in missing_paths)
    raise FileNotFoundError(f"Required cleaned CSV file(s) are missing:\n{missing_list}")

raw_datasets = {}
for dataset_name, path in dataset_paths.items():
    dataset = pd.read_csv(path)
    if dataset.empty or dataset.shape[1] == 0:
        raise ValueError(
            f"{dataset_name} did not load with actual rows and columns: "
            f"{path} (shape {dataset.shape})"
        )
    raw_datasets[dataset_name] = dataset
    print(f"{dataset_name}: {path.relative_to(project_root)} -> {dataset.shape}")

Infrastructure: data\cleaned\infrastructure_state_year.csv -> (74, 11)
Budget: data\cleaned\nhm_budget_long_corrected.csv -> (108, 5)
HMIS: data\cleaned\hmis_long_normalized.csv -> (101840, 9)


## Infrastructure Validation

Check completeness, exact duplicates, State-Year uniqueness, column-name consistency, numeric parseability, and basic State/Year validity. Column names are normalized only in an in-memory copy; missing values and invalid entries are reported, never filled.

In [2]:
from IPython.display import display

validation_results = {}
infrastructure = raw_datasets["Infrastructure"].copy()
original_columns = list(infrastructure.columns)
standardized_columns = (
    infrastructure.columns.astype(str)
    .str.strip()
    .str.replace(r"[^0-9A-Za-z]+", "_", regex=True)
    .str.strip("_")
)
if standardized_columns.has_duplicates:
    raise ValueError("Infrastructure column normalization would create duplicate names.")
infrastructure.columns = standardized_columns
renamed_columns = {
    old: new
    for old, new in zip(original_columns, infrastructure.columns)
    if old != new
}
print(f"Shape: {infrastructure.shape}")
print(f"In-memory column-name changes: {renamed_columns or 'none needed'}")
print(f"Columns: {list(infrastructure.columns)}")

infrastructure_missing = infrastructure.isna().sum().rename("missing_count").to_frame()
infrastructure_missing["missing_percent"] = (
    infrastructure.isna().mean().mul(100).round(2)
)
print("Missing values by column:")
display(infrastructure_missing)
infrastructure_duplicate_rows = int(infrastructure.duplicated().sum())
print(f"Exact duplicate rows: {infrastructure_duplicate_rows}")

required_infrastructure_columns = {"State", "Year"}
missing_infrastructure_columns = sorted(
    required_infrastructure_columns.difference(infrastructure.columns)
)
if missing_infrastructure_columns:
    print(f"Missing required columns: {missing_infrastructure_columns}")
    state_year_duplicate_rows = None
    invalid_state_rows = None
    invalid_year_rows = None
else:
    state_values = infrastructure["State"].astype("string")
    invalid_state_mask = state_values.isna() | state_values.str.strip().eq("")
    invalid_state_rows = int(invalid_state_mask.sum())
    year_values = infrastructure["Year"].astype("string")
    valid_year_mask = year_values.str.fullmatch(r"\d{4}-\d{2,4}", na=False)
    invalid_year_mask = year_values.notna() & ~valid_year_mask
    invalid_year_rows = int(invalid_year_mask.sum())
    print(f"Invalid or blank State rows: {invalid_state_rows}")
    print(f"Year values: {sorted(year_values.dropna().unique().tolist())}")
    print(f"Year values outside YYYY-YY/YYY format: {invalid_year_rows}")
    state_year_duplicate_mask = infrastructure.duplicated(
        subset=["State", "Year"], keep=False
    )
    state_year_duplicate_rows = int(state_year_duplicate_mask.sum())
    print(f"Rows in duplicate State-Year combinations: {state_year_duplicate_rows}")
    if state_year_duplicate_rows:
        display(
            infrastructure.loc[
                state_year_duplicate_mask, ["State", "Year"]
            ].drop_duplicates()
        )

numeric_columns = [
    column for column in infrastructure.columns if column not in {"State", "Year"}
]
numeric_parsed = infrastructure[numeric_columns].apply(pd.to_numeric, errors="coerce")
numeric_parse_failures = int(
    (infrastructure[numeric_columns].notna() & numeric_parsed.isna()).sum().sum()
)
numeric_check = pd.DataFrame(
    {
        "source_dtype": infrastructure[numeric_columns].dtypes.astype(str),
        "non_numeric_nonmissing_values": (
            infrastructure[numeric_columns].notna() & numeric_parsed.isna()
        ).sum(),
    }
)
print("Numeric-column validation:")
display(numeric_check)

infrastructure_missing_cells = int(infrastructure.isna().sum().sum())
infrastructure_issues = [
    infrastructure_missing_cells > 0,
    infrastructure_duplicate_rows > 0,
    bool(missing_infrastructure_columns),
    invalid_state_rows not in (None, 0),
    invalid_year_rows not in (None, 0),
    numeric_parse_failures > 0,
    state_year_duplicate_rows not in (None, 0),
]
infrastructure_status = "FAIL" if missing_infrastructure_columns else (
    "REVIEW" if any(infrastructure_issues) else "PASS"
)
validation_results["Infrastructure"] = {
    "Rows": len(infrastructure),
    "Columns": len(infrastructure.columns),
    "Missing Cells": infrastructure_missing_cells,
    "Duplicate Rows": infrastructure_duplicate_rows,
    "Validation Status": infrastructure_status,
}
print(f"Infrastructure validation status: {infrastructure_status}")

Shape: (74, 11)
In-memory column-name changes: none needed
Columns: ['State', 'Year', 'SHCs', 'PHCs', 'District_Hospitals', 'PHC_Beds', 'CHC_Beds', 'SDH_Beds', 'District_Hospital_Beds', 'Medical_College_Beds', 'Total_Beds']
Missing values by column:


,missing_count,missing_percent
State,0,0.00
Year,0,0.00
SHCs,37,50.00
PHCs,37,50.00
District_Hospitals,37,50.00
PHC_Beds,37,50.00
CHC_Beds,38,51.35
SDH_Beds,45,60.81
District_Hospital_Beds,37,50.00
Medical_College_Beds,41,55.41


Exact duplicate rows: 0
Invalid or blank State rows: 0
Year values: ['2021-22', '2022-23']
Year values outside YYYY-YY/YYY format: 0
Rows in duplicate State-Year combinations: 0
Numeric-column validation:


,source_dtype,non_numeric_nonmissing_values
SHCs,float64,0
PHCs,float64,0
District_Hospitals,float64,0
PHC_Beds,float64,0
CHC_Beds,float64,0
SDH_Beds,float64,0
District_Hospital_Beds,float64,0
Medical_College_Beds,float64,0
Total_Beds,float64,0


Infrastructure validation status: REVIEW


In [5]:
print(f"Distinct non-null State values: {sorted(infrastructure['State'].dropna().unique().tolist())}")
print(
    "State check is limited to nonblank values; no external canonical State/UT "
    "reference list was provided for membership validation."
)

Distinct non-null State values: ['All India/Total', 'Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli and Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Total', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']
State check is limited to nonblank values; no external canonical State/UT reference list was provided for membership validation.


## Budget Validation

Validate required columns, missingness, exact and State-Year duplicates, numeric parseability, negative values, and the stored expenditure-to-release percentage. The percentage is compared with `Expenditure / Central_Release * 100` only where both inputs are present and the release is nonzero; values above 100% are not treated as impossible by themselves.

In [3]:
budget = raw_datasets["Budget"].copy()
print(f"Shape: {budget.shape}")
budget_missing = budget.isna().sum().rename("missing_count").to_frame()
budget_missing["missing_percent"] = budget.isna().mean().mul(100).round(2)
print("Missing values by column:")
display(budget_missing)
budget_duplicate_rows = int(budget.duplicated().sum())
print(f"Exact duplicate rows: {budget_duplicate_rows}")

budget_required_columns = {
    "State",
    "Year",
    "Central_Release",
    "Expenditure",
    "Expenditure_to_Release_Pct",
}
missing_budget_columns = sorted(budget_required_columns.difference(budget.columns))
if missing_budget_columns:
    print(f"Missing required columns: {missing_budget_columns}")
    budget_state_year_duplicate_rows = None
    numeric_parse_failures = None
    negative_counts = {}
    percentage_mismatches = None
else:
    budget_state_year_mask = budget.duplicated(
        subset=["State", "Year"], keep=False
    )
    budget_state_year_duplicate_rows = int(budget_state_year_mask.sum())
    print(
        "Rows in duplicate State-Year combinations: "
        f"{budget_state_year_duplicate_rows}"
    )
    if budget_state_year_duplicate_rows:
        display(
            budget.loc[
                budget_state_year_mask, ["State", "Year"]
            ].drop_duplicates()
        )

    budget_numeric_columns = [
        "Central_Release",
        "Expenditure",
        "Expenditure_to_Release_Pct",
    ]
    budget_numeric = budget[budget_numeric_columns].apply(
        pd.to_numeric, errors="coerce"
    )
    numeric_parse_failures = int(
        (budget[budget_numeric_columns].notna() & budget_numeric.isna())
        .sum()
        .sum()
    )
    numeric_check = pd.DataFrame(
        {
            "source_dtype": budget[budget_numeric_columns].dtypes.astype(str),
            "non_numeric_nonmissing_values": (
                budget[budget_numeric_columns].notna() & budget_numeric.isna()
            ).sum(),
        }
    )
    print("Numeric-column validation:")
    display(numeric_check)

    negative_counts = {
        column: int(budget_numeric[column].lt(0).sum())
        for column in budget_numeric_columns
    }
    print(f"Negative values by numeric column: {negative_counts}")
    comparable = (
        budget_numeric["Central_Release"].notna()
        & budget_numeric["Expenditure"].notna()
        & budget_numeric["Expenditure_to_Release_Pct"].notna()
        & budget_numeric["Central_Release"].ne(0)
    )
    expected_percentage = (
        budget_numeric.loc[comparable, "Expenditure"]
        / budget_numeric.loc[comparable, "Central_Release"]
        * 100
    )
    actual_percentage = budget_numeric.loc[
        comparable, "Expenditure_to_Release_Pct"
    ]
    percentage_mismatch_mask = ~np.isclose(
        actual_percentage, expected_percentage, rtol=1e-6, atol=1e-6
    )
    percentage_mismatches = int(percentage_mismatch_mask.sum())
    print(
        "Expenditure-to-release percentage: "
        f"{int(comparable.sum())} comparable rows, "
        f"{percentage_mismatches} formula mismatches."
    )
    unverifiable_percentage_rows = int((~comparable).sum())
    if unverifiable_percentage_rows:
        print(
            "Rows where the percentage cannot be recomputed because an input is "
            f"missing or release is zero: {unverifiable_percentage_rows}"
        )

budget_missing_cells = int(budget.isna().sum().sum())
budget_issues = [
    budget_missing_cells > 0,
    budget_duplicate_rows > 0,
    budget_state_year_duplicate_rows not in (None, 0),
    numeric_parse_failures not in (None, 0),
    any(count > 0 for count in negative_counts.values()),
    percentage_mismatches not in (None, 0),
]
budget_status = "FAIL" if missing_budget_columns else (
    "REVIEW" if any(budget_issues) else "PASS"
)
validation_results["Budget"] = {
    "Rows": len(budget),
    "Columns": len(budget.columns),
    "Missing Cells": budget_missing_cells,
    "Duplicate Rows": budget_duplicate_rows,
    "Validation Status": budget_status,
}
print(f"Budget validation status: {budget_status}")

Shape: (108, 5)
Missing values by column:


,missing_count,missing_percent
State,0,0.0
Year,0,0.0
Central_Release,0,0.0
Expenditure,0,0.0
Expenditure_to_Release_Pct,0,0.0


Exact duplicate rows: 0
Rows in duplicate State-Year combinations: 0
Numeric-column validation:


,source_dtype,non_numeric_nonmissing_values
Central_Release,float64,0
Expenditure,float64,0
Expenditure_to_Release_Pct,float64,0


Negative values by numeric column: {'Central_Release': 0, 'Expenditure': 0, 'Expenditure_to_Release_Pct': 0}
Expenditure-to-release percentage: 108 comparable rows, 0 formula mismatches.
Budget validation status: PASS


## HMIS Validation

Review completeness and exact duplicates, inspect `Observation_Category`, compare `Value` with `Value_Numeric`, and test `Reporting_Period` for parseable dates or year labels. Check one composite observation key built from available reporting dimensions; repeated keys are reported, not removed. No date or year is inferred.

In [4]:
hmis = raw_datasets["HMIS"].copy()
print(f"Shape: {hmis.shape}")
hmis_missing = hmis.isna().sum().rename("missing_count").to_frame()
hmis_missing["missing_percent"] = hmis.isna().mean().mul(100).round(2)
print("Missing values by column:")
display(hmis_missing)
hmis_missing_cells = int(hmis.isna().sum().sum())
hmis_duplicate_rows = int(hmis.duplicated().sum())
print(f"Exact duplicate rows: {hmis_duplicate_rows}")

if "Observation_Category" in hmis.columns:
    category_counts = hmis["Observation_Category"].value_counts(dropna=False)
    print("Observation_Category values and row counts:")
    display(category_counts.rename("row_count").to_frame())
else:
    category_counts = pd.Series(dtype="int64")
    print("Observation_Category column is missing.")

required_value_columns = {"Value", "Value_Numeric"}
missing_value_columns = sorted(required_value_columns.difference(hmis.columns))
if missing_value_columns:
    print(f"Missing value columns: {missing_value_columns}")
    value_parse_failures = None
    value_mismatches = None
    unpaired_values = None
else:
    value_as_numeric = pd.to_numeric(hmis["Value"], errors="coerce")
    value_numeric_as_numeric = pd.to_numeric(
        hmis["Value_Numeric"], errors="coerce"
    )
    value_parse_failures = int(
        (hmis["Value"].notna() & value_as_numeric.isna()).sum()
        + (hmis["Value_Numeric"].notna() & value_numeric_as_numeric.isna()).sum()
    )
    paired_values = value_as_numeric.notna() & value_numeric_as_numeric.notna()
    value_mismatches = int(
        (~np.isclose(
            value_as_numeric.loc[paired_values],
            value_numeric_as_numeric.loc[paired_values],
            rtol=1e-9,
            atol=1e-9,
        )).sum()
    )
    unpaired_values = int(
        hmis["Value"].isna().ne(hmis["Value_Numeric"].isna()).sum()
    )
    print(
        f"Value/Value_Numeric: {int(paired_values.sum())} comparable rows, "
        f"{value_mismatches} mismatches, {value_parse_failures} non-numeric "
        f"non-missing values, {unpaired_values} rows with only one value missing."
    )

if "Reporting_Period" in hmis.columns:
    reporting_period_values = hmis["Reporting_Period"].dropna().unique().tolist()
    parsed_reporting_dates = pd.to_datetime(
        hmis["Reporting_Period"], errors="coerce", format="mixed"
    )
    date_rows = int(parsed_reporting_dates.notna().sum())
    year_label_rows = int(
        hmis["Reporting_Period"]
        .astype("string")
        .str.contains(r"\b(?:19|20)\d{2}\b", regex=True, na=False)
        .sum()
    )
    print(f"Reporting_Period distinct values: {reporting_period_values}")
    print(
        f"Reporting_Period parseable date rows: {date_rows}; "
        f"rows containing a 4-digit year label: {year_label_rows}."
    )
else:
    reporting_period_values = []
    date_rows = None
    year_label_rows = None
    print("Reporting_Period column is missing; dates/years cannot be checked.")

hmis_key_columns = [
    column
    for column in [
        "Indicator",
        "S.No.",
        "Parameters",
        "Type",
        "State",
        "Observation_Category",
        "Reporting_Period",
    ]
    if column in hmis.columns
]
if len(hmis_key_columns) < 2:
    hmis_key_duplicate_rows = None
    hmis_key_duplicate_groups = None
    reliable_hmis_key = False
    print(
        "No usable composite observation-key candidate: fewer than two "
        "reporting-dimension columns are available."
    )
else:
    hmis_key_duplicate_mask = hmis.duplicated(
        subset=hmis_key_columns, keep=False
    )
    hmis_key_duplicate_rows = int(hmis_key_duplicate_mask.sum())
    hmis_key_duplicate_groups = int(
        hmis.loc[hmis_key_duplicate_mask, hmis_key_columns]
        .drop_duplicates()
        .shape[0]
    )
    reliable_hmis_key = hmis_key_duplicate_rows == 0
    print(f"Candidate observation key columns: {hmis_key_columns}")
    print(
        f"Duplicate candidate-key groups: {hmis_key_duplicate_groups}; "
        f"rows involved: {hmis_key_duplicate_rows}."
    )
    if not reliable_hmis_key:
        print("This candidate is not unique; no reliable unique observation key was established.")

hmis_issues = [
    hmis_missing_cells > 0,
    hmis_duplicate_rows > 0,
    bool(missing_value_columns),
    value_parse_failures not in (None, 0),
    value_mismatches not in (None, 0),
    unpaired_values not in (None, 0),
    date_rows in (None, 0),
    year_label_rows in (None, 0),
    hmis_key_duplicate_rows not in (None, 0),
]
hmis_status = "FAIL" if missing_value_columns else (
    "REVIEW" if any(hmis_issues) else "PASS"
)
validation_results["HMIS"] = {
    "Rows": len(hmis),
    "Columns": len(hmis.columns),
    "Missing Cells": hmis_missing_cells,
    "Duplicate Rows": hmis_duplicate_rows,
    "Validation Status": hmis_status,
}
print(f"HMIS validation status: {hmis_status}")

Shape: (101840, 9)
Missing values by column:


,missing_count,missing_percent
Indicator,0,0.00
S.No.,0,0.00
Parameters,0,0.00
Type,0,0.00
State,0,0.00
Observation_Category,0,0.00
Value,44862,44.05
Value_Numeric,44862,44.05
Reporting_Period,0,0.00


Exact duplicate rows: 1539
Observation_Category values and row counts:


,row_count
Observation_Category,
Total [(A+B) or (C+D)],20368
Public [A],20368
Private [B],20368
Urban [C],20368
Rural [D],20368


Value/Value_Numeric: 56978 comparable rows, 0 mismatches, 0 non-numeric non-missing values, 0 rows with only one value missing.
Reporting_Period distinct values: ['Source snapshot (date not inferred)']
Reporting_Period parseable date rows: 0; rows containing a 4-digit year label: 0.
Candidate observation key columns: ['Indicator', 'S.No.', 'Parameters', 'Type', 'State', 'Observation_Category', 'Reporting_Period']
Duplicate candidate-key groups: 2680; rows involved: 5360.
This candidate is not unique; no reliable unique observation key was established.
HMIS validation status: REVIEW


## Final Validation Summary

`PASS` means the implemented checks found no issues. `REVIEW` flags missing values, duplicates, date/key limitations, or other findings that need a project decision; no rows are removed or imputed here. `FAIL` means required columns for a check are absent. This summary describes validation only and does not write changes back to the source CSVs.

In [7]:
summary_columns = [
    "Dataset",
    "Rows",
    "Columns",
    "Missing Cells",
    "Duplicate Rows",
    "Validation Status",
]
validation_summary = (
    pd.DataFrame.from_dict(validation_results, orient="index")
    .rename_axis("Dataset")
    .reset_index()
)
validation_summary = validation_summary[summary_columns]
display(validation_summary)

          Dataset    Rows  Columns  Missing Cells  Duplicate Rows Validation Status0  Infrastructure      74       11            346              0           REVIEW1          Budget     108        5              0              0             PASS2            HMIS  101840        9          89724           1539           REVIEW